
# Learning the Gerchberg-Saxton (GS) algorithm — naive interactive notebook

This notebook is meant for intuition, not performance.

We simulate a phase-only SLM in the simplest possible way:

1. Start at the SLM plane with a known input amplitude, usually a flat-top or Gaussian beam.
2. Add a random phase mask.
3. Propagate to the Fourier plane using `FFT2`.
4. Replace the Fourier-plane amplitude with the desired target amplitude, but keep the current phase.
5. Propagate back using `IFFT2`.
6. Replace the SLM-plane amplitude with the original input amplitude, but keep the updated phase.
7. Repeat.

At the end, the usable SLM mask is the phase in the SLM plane.

For the Meadowlark P512 style workflow, this notebook uses a default grid of **512 × 512** and returns a phase mask in **[0, 2π)**. There is also an optional helper cell to export a 16-bit DVI-style bitmap.



## 0. Imports

Run this cell first. The interactive widgets need `ipywidgets`. In JupyterLab/Notebook this usually works out of the box; in VS Code notebooks it may ask you to install/enable widgets.


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

try:
    import ipywidgets as widgets
    WIDGETS_AVAILABLE = True
except Exception as exc:
    WIDGETS_AVAILABLE = False
    print("ipywidgets is not available. The non-interactive cells will still work.")
    print(exc)

plt.rcParams["figure.figsize"] = (6, 5)
plt.rcParams["image.cmap"] = "gray"



## 1. Helper functions

These functions intentionally avoid advanced tricks:

- no weighting,
- no MRAF,
- no camera feedback,
- no measured input beam correction,
- no zero-order suppression,
- no aperture optimization,
- no fancy initialization.

This is the most basic GS loop.


In [ ]:

def normalize01(a, eps=1e-12):
    """Normalize array to [0, 1] for plotting."""
    a = np.asarray(a, dtype=float)
    amin, amax = np.nanmin(a), np.nanmax(a)
    return (a - amin) / (amax - amin + eps)


def make_xy_grid(N=512):
    """Coordinate grid on [-1, 1] x [-1, 1]."""
    x = np.linspace(-1, 1, N)
    X, Y = np.meshgrid(x, x)
    R = np.sqrt(X**2 + Y**2)
    TH = np.arctan2(Y, X)
    return X, Y, R, TH


def make_input_amplitude(N=512, kind="gaussian", waist=0.45):
    """SLM-plane input amplitude: simple flat-top or Gaussian beam."""
    X, Y, R, TH = make_xy_grid(N)
    if kind == "flat":
        amp = np.ones((N, N), dtype=float)
    elif kind == "circular aperture":
        amp = (R < 0.85).astype(float)
    elif kind == "gaussian":
        amp = np.exp(-(R / waist)**2)
    else:
        raise ValueError(f"Unknown input amplitude kind: {kind}")
    return amp / np.sqrt(np.sum(amp**2))


def make_target_intensity(N=512, kind="two spots", spot_sigma=0.035):
    """Fourier-plane target intensity patterns for playing with GS."""
    X, Y, R, TH = make_xy_grid(N)
    I = np.zeros((N, N), dtype=float)

    def add_gaussian(cx, cy, sigma=spot_sigma, weight=1.0):
        return weight * np.exp(-((X - cx)**2 + (Y - cy)**2) / (2 * sigma**2))

    if kind == "one spot":
        I = add_gaussian(0.0, 0.0)
    elif kind == "two spots":
        I = add_gaussian(-0.35, 0.0) + add_gaussian(0.35, 0.0)
    elif kind == "four spots":
        for cx in [-0.35, 0.35]:
            for cy in [-0.35, 0.35]:
                I += add_gaussian(cx, cy)
    elif kind == "ring / donut":
        I = np.exp(-((R - 0.38)**2) / (2 * spot_sigma**2))
    elif kind == "HG10 intensity":
        w = 0.38
        A = X * np.exp(-(X**2 + Y**2) / w**2)
        I = A**2
    elif kind == "HG11 intensity":
        w = 0.38
        A = X * Y * np.exp(-(X**2 + Y**2) / w**2)
        I = A**2
    elif kind == "letter E":
        I = np.zeros_like(X)
        I[(X > -0.55) & (X < -0.45) & (Y > -0.45) & (Y < 0.45)] = 1
        I[(X > -0.55) & (X < 0.35) & (Y > 0.32) & (Y < 0.45)] = 1
        I[(X > -0.55) & (X < 0.20) & (Y > -0.06) & (Y < 0.06)] = 1
        I[(X > -0.55) & (X < 0.35) & (Y > -0.45) & (Y < -0.32)] = 1
        for _ in range(3):
            I = (I + np.roll(I, 1, 0) + np.roll(I, -1, 0) + np.roll(I, 1, 1) + np.roll(I, -1, 1)) / 5
    elif kind == "random sparse spots":
        rng = np.random.default_rng(1)
        for _ in range(12):
            cx, cy = rng.uniform(-0.55, 0.55, size=2)
            weight = rng.uniform(0.4, 1.0)
            I += add_gaussian(cx, cy, sigma=spot_sigma, weight=weight)
    else:
        raise ValueError(f"Unknown target kind: {kind}")

    I = np.clip(I, 0, None)
    I /= I.sum() if I.sum() > 0 else 1.0
    return I


def centered_fft2(u):
    """Centered Fourier transform, normalized only for convenient numerics."""
    return np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(u), norm="ortho"))


def centered_ifft2(U):
    """Inverse of centered_fft2."""
    return np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(U), norm="ortho"))


def gs_naive(input_amp, target_amp, n_iter=30, seed=0, store_every=1):
    """
    Naive Gerchberg-Saxton.

    input_amp: fixed SLM-plane amplitude.
    target_amp: desired Fourier-plane amplitude.
    n_iter: number of GS iterations.
    store_every: save states every this many iterations.
    """
    rng = np.random.default_rng(seed)
    phase = rng.uniform(0, 2*np.pi, size=input_amp.shape)
    u = input_amp * np.exp(1j * phase)

    history = []
    errors = []

    for k in range(n_iter + 1):
        U = centered_fft2(u)
        current_amp = np.abs(U)
        current_intensity = current_amp**2
        err = np.sqrt(np.mean((normalize01(current_amp) - normalize01(target_amp))**2))
        errors.append(err)

        if k % store_every == 0 or k == n_iter:
            history.append({
                "iteration": k,
                "slm_phase": np.mod(np.angle(u), 2*np.pi),
                "fourier_intensity": current_intensity.copy(),
                "error": err,
            })

        if k == n_iter:
            break

        U_new = target_amp * np.exp(1j * np.angle(U))
        u_back = centered_ifft2(U_new)
        u = input_amp * np.exp(1j * np.angle(u_back))

    return {
        "phase": np.mod(np.angle(u), 2*np.pi),
        "history": history,
        "errors": np.array(errors),
    }



## 2. Static first run

This cell runs a small example without widgets. It is good for checking that everything works.


In [ ]:

N = 512
input_amp = make_input_amplitude(N=N, kind="gaussian", waist=0.45)
target_I = make_target_intensity(N=N, kind="four spots", spot_sigma=0.035)
target_amp = np.sqrt(target_I)

result = gs_naive(input_amp, target_amp, n_iter=30, seed=3, store_every=5)

print("Stored iterations:", [h["iteration"] for h in result["history"]])
print("Final amplitude RMSE-like error:", result["errors"][-1])



## 3. Plot target, final result, and error curve


In [ ]:

def plot_summary(input_amp, target_I, result):
    final_I = result["history"][-1]["fourier_intensity"]
    final_phase = result["phase"]

    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    axes[0].imshow(input_amp)
    axes[0].set_title("SLM input amplitude")
    axes[0].axis("off")

    axes[1].imshow(target_I)
    axes[1].set_title("Target intensity")
    axes[1].axis("off")

    axes[2].imshow(normalize01(final_I))
    axes[2].set_title("Final Fourier intensity")
    axes[2].axis("off")

    axes[3].imshow(final_phase, cmap="twilight", vmin=0, vmax=2*np.pi)
    axes[3].set_title("Final SLM phase")
    axes[3].axis("off")

    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(6, 4))
    plt.plot(result["errors"], marker=".")
    plt.xlabel("Iteration")
    plt.ylabel("Amplitude RMSE-like error")
    plt.title("Naive GS convergence diagnostic")
    plt.grid(True, alpha=0.3)
    plt.show()

plot_summary(input_amp, target_I, result)



## 4. Print all stored iterations side by side

This is the main “learning” visualization. Each column is one saved iteration.

Top row: Fourier-plane intensity.  
Bottom row: SLM phase that generated it.


In [ ]:
def plot_iterations_side_by_side(history, max_cols=None):
    if max_cols is not None:
        history = history[:max_cols]

    n = len(history)
    if n == 0:
        print("No iterations stored.")
        return

    # Keep figures readable and avoid browser freezes from enormous side-by-side outputs.
    fig_width = min(3.0*n, 36)
    fig, axes = plt.subplots(2, n, figsize=(fig_width, 6), squeeze=False)

    for j, h in enumerate(history):
        axes[0, j].imshow(normalize01(h["fourier_intensity"]))
        axes[0, j].set_title(f"Iter {h['iteration']}\nerr={h['error']:.3f}")
        axes[0, j].axis("off")

        axes[1, j].imshow(h["slm_phase"], cmap="twilight", vmin=0, vmax=2*np.pi)
        axes[1, j].set_title("SLM phase")
        axes[1, j].axis("off")

    plt.tight_layout()
    plt.show()



## 5. Interactive GS playground

Change target pattern, number of iterations, input beam type, random seed, and how often iterations are stored.

For big iteration counts, do **not** store every single iteration unless you actually want a huge side-by-side plot.


In [ ]:
def pick_evenly_spaced(history, max_items=12):
    """Return at most max_items entries, evenly spaced, always including first and last."""
    if len(history) <= max_items:
        return history
    idx = np.linspace(0, len(history) - 1, max_items).round().astype(int)
    idx = np.unique(idx)
    return [history[i] for i in idx]


def interactive_gs():
    if not WIDGETS_AVAILABLE:
        print("ipywidgets is unavailable, so the interactive playground cannot be displayed.")
        return

    target_dropdown = widgets.Dropdown(
        options=["one spot", "two spots", "four spots", "ring / donut", "HG10 intensity", "HG11 intensity", "letter E", "random sparse spots"],
        value="four spots",
        description="Target:",
        layout=widgets.Layout(width="260px"),
    )
    input_dropdown = widgets.Dropdown(
        options=["gaussian", "flat", "circular aperture"],
        value="gaussian",
        description="Input:",
        layout=widgets.Layout(width="260px"),
    )

    # 256 is much nicer for interactive learning. Use 512 when you actually want SLM-sized masks.
    N_dropdown = widgets.Dropdown(
        options=[128, 256, 512],
        value=256,
        description="Grid N:",
        layout=widgets.Layout(width="180px"),
    )
    n_slider = widgets.IntSlider(value=30, min=1, max=150, step=1, description="Iterations:", continuous_update=False)
    store_slider = widgets.IntSlider(value=5, min=1, max=25, step=1, description="Store every:", continuous_update=False)
    max_panels_slider = widgets.IntSlider(value=12, min=4, max=20, step=1, description="Max panels:", continuous_update=False)
    waist_slider = widgets.FloatSlider(value=0.45, min=0.15, max=1.00, step=0.01, description="Waist:", continuous_update=False)
    sigma_slider = widgets.FloatSlider(value=0.035, min=0.01, max=0.12, step=0.005, description="Spot sigma:", continuous_update=False)
    seed_slider = widgets.IntSlider(value=3, min=0, max=999, step=1, description="Seed:", continuous_update=False)
    run_button = widgets.Button(description="Run GS once", button_style="primary")
    out = widgets.Output()

    controls = widgets.VBox([
        widgets.HTML("<b>Nothing runs until you click the button.</b> This avoids slow automatic re-runs while editing sliders."),
        widgets.HBox([target_dropdown, input_dropdown, N_dropdown]),
        widgets.HBox([n_slider, store_slider, max_panels_slider]),
        widgets.HBox([waist_slider, sigma_slider, seed_slider]),
        run_button,
    ])

    def run(_=None):
        run_button.disabled = True
        run_button.description = "Running..."
        try:
            with out:
                clear_output(wait=True)
                N = int(N_dropdown.value)
                input_amp = make_input_amplitude(N=N, kind=input_dropdown.value, waist=waist_slider.value)
                target_I = make_target_intensity(N=N, kind=target_dropdown.value, spot_sigma=sigma_slider.value)
                target_amp = np.sqrt(target_I)
                result = gs_naive(input_amp, target_amp, n_iter=n_slider.value, seed=seed_slider.value, store_every=store_slider.value)

                history_to_plot = pick_evenly_spaced(result["history"], max_items=max_panels_slider.value)

                print(f"Target: {target_dropdown.value}")
                print(f"Input: {input_dropdown.value}")
                print(f"Grid: {N} x {N}")
                print(f"Stored iterations: {[h['iteration'] for h in result['history']]}")
                print(f"Displayed iterations: {[h['iteration'] for h in history_to_plot]}")
                print(f"Final error: {result['errors'][-1]:.5f}")
                plot_summary(input_amp, target_I, result)
                plot_iterations_side_by_side(history_to_plot)

                globals()["latest_input_amp"] = input_amp
                globals()["latest_target_I"] = target_I
                globals()["latest_gs_result"] = result
        finally:
            run_button.disabled = False
            run_button.description = "Run GS once"

    run_button.on_click(run)
    display(controls, out)

interactive_gs()



## 6. What exactly is happening in one GS iteration?

This cell shows the data flow for a single iteration using the current field.


In [ ]:

def show_one_iteration(input_amp, target_amp, seed=0):
    rng = np.random.default_rng(seed)
    u0 = input_amp * np.exp(1j * rng.uniform(0, 2*np.pi, input_amp.shape))

    U0 = centered_fft2(u0)
    U1 = target_amp * np.exp(1j * np.angle(U0))
    u1_back = centered_ifft2(U1)
    u1 = input_amp * np.exp(1j * np.angle(u1_back))
    U_after = centered_fft2(u1)

    titles = [
        "1. Initial SLM phase",
        "2. Fourier intensity before constraint",
        "3. Target amplitude inserted",
        "4. Back plane phase after IFFT",
        "5. Fourier intensity after one full iteration",
    ]
    images = [
        np.mod(np.angle(u0), 2*np.pi),
        normalize01(np.abs(U0)**2),
        normalize01(np.abs(U1)**2),
        np.mod(np.angle(u1_back), 2*np.pi),
        normalize01(np.abs(U_after)**2),
    ]
    cmaps = ["twilight", "gray", "gray", "twilight", "gray"]

    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    for ax, im, title, cmap in zip(axes, images, titles, cmaps):
        if cmap == "twilight":
            ax.imshow(im, cmap=cmap, vmin=0, vmax=2*np.pi)
        else:
            ax.imshow(im, cmap=cmap)
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

show_one_iteration(input_amp, target_amp, seed=3)



## 7. Export phase for a 512 × 512 phase-only SLM

The final phase is in radians in `[0, 2π)`. A common software-side representation is a 16-bit integer image where

\[
0 \rightarrow 0, \qquad 2\pi \rightarrow 65535.
\]

For Meadowlark DVI-style 24-bit bitmap loading, the 16-bit value is commonly split across color channels:

- green = most significant 8 bits,
- red = least significant 8 bits,
- blue = ignored.

This helper saves both:

1. a plain 16-bit grayscale PNG/TIFF-friendly array, and  
2. a 24-bit BMP with the red/green channel packing.

Use your own calibrated LUT in the Meadowlark software when actually displaying the pattern.


In [ ]:

from PIL import Image
from pathlib import Path


def phase_to_uint16(phase_rad):
    """Map phase [0, 2pi) to uint16 [0, 65535]."""
    phase_wrapped = np.mod(phase_rad, 2*np.pi)
    return np.uint16(np.round(phase_wrapped * 65535 / (2*np.pi)))


def save_meadowlark_dvi_bmp(phase_rad, path="gs_phase_mask_dvi.bmp"):
    """
    Save phase as a 24-bit BMP for Meadowlark DVI-style 16-bit channel packing.

    Pixel value v is split as:
      green = high byte = v >> 8
      red   = low byte  = v & 255
      blue  = 0
    """
    v = phase_to_uint16(phase_rad)
    red = (v & 255).astype(np.uint8)
    green = (v >> 8).astype(np.uint8)
    blue = np.zeros_like(red, dtype=np.uint8)
    rgb = np.dstack([red, green, blue])
    Image.fromarray(rgb, mode="RGB").save(path)
    print(f"Saved: {Path(path).resolve()}")


def save_phase_uint16_tif(phase_rad, path="gs_phase_mask_uint16.tif"):
    v = phase_to_uint16(phase_rad)
    Image.fromarray(v).save(path)
    print(f"Saved: {Path(path).resolve()}")

phase_to_export = globals().get("latest_gs_result", result)["phase"]
save_phase_uint16_tif(phase_to_export, "gs_phase_mask_uint16.tif")
save_meadowlark_dvi_bmp(phase_to_export, "gs_phase_mask_dvi.bmp")



## 8. Things to notice

1. **GS does not create arbitrary complex fields with a phase-only SLM.** It tries to satisfy two constraints that may be incompatible: fixed SLM-plane amplitude and desired Fourier-plane amplitude.
2. **The target intensity is not the same thing as the target complex field.** This notebook constrains only the Fourier amplitude; the Fourier phase is allowed to become whatever GS finds.
3. **Zero order is ignored here.** Real SLMs have unmodulated/reflection/pixel-structure contributions, so experiments often use a carrier grating and spatial filtering.
4. **A real beam is not perfectly Gaussian or flat.** A better practical version uses a measured input amplitude.
5. **A real SLM has LUT, wavefront, polarization, fill-factor, and diffraction-order issues.** This notebook is deliberately the clean mathematical core before all that lab reality is added.
